# Tiger HLM Setup
Generate lookups, YAML, and SLURM files for runoff + routing runs.

In [ ]:
from tiger_hlm_setup import generate_lookup, get_forcing_characteristics, setup_longterm, setup_forecast, setup_spinup

## 1. Generate Forcing Lookups

In [ ]:
params_csv  = '/path/to/parameters/CONUS_West_runoff_params.csv'
pr_ncfile   = '/path/to/precip/product_20020101_20020104.nc'
t2_ncfile   = '/path/to/temperature/AORC_t2m_daily_20020101_20020107.nc'
lookup_dir  = '/path/to/lookups'

# Inspect forcing files to get variable names, dims, resolution
pr_varname, pr_dims, pr_resolution = get_forcing_characteristics(pr_ncfile)
t2_varname, t2_dims, t2_resolution = get_forcing_characteristics(t2_ncfile)
print(pr_varname, pr_dims, pr_resolution)
print(t2_varname, t2_dims, t2_resolution)

# Build lookup CSVs  (flip_dims=True for IMERG)
generate_lookup(pr_ncfile, params_csv, f'{lookup_dir}/AORC_West_lookup_pr.csv')
generate_lookup(t2_ncfile, params_csv, f'{lookup_dir}/AORC_West_lookup_t2m.csv')

## 2. Long-term Simulation (2002–2019)

In [ ]:
runoff_inputs = {
    'params_dir':       '/path/to/parameters',
    'params_csv':       'CONUS_West_runoff_params.csv',
    'forcings_dir':     '/path/to/forcings',
    'pr_file_pattern':  'Products/AORC/monthly/product_{start}_{end}.nc',
    'pr_varname':       pr_varname,
    'pr_resolution':    pr_resolution,
    'pr_dims':          pr_dims,
    't2_file_pattern':  'temperature/AORC_t2m_{start}_{end}.nc',
    't2_varname':       t2_varname,
    't2_dims':          t2_dims,
    'lookup_dir':       lookup_dir,
    'lookup_pr_csv':    'AORC_West_lookup_pr.csv',
    'lookup_t2m_csv':   'AORC_West_lookup_t2m.csv',
}

routing_inputs = {
    'params':    '/path/to/parameters/CONUS_West_routing_params.csv',
    'sav_path':  '/path/to/West_sav.csv',
}

# Optional: override SLURM defaults
slurm_cfg = {
    'account':  'gvillari',
    'email':    'user@princeton.edu',
}

setup_longterm(
    proj_root          = '/scratch/gpfs/user/MyProject',
    start_year         = 2002,
    end_year           = 2019,
    runoff_inputs      = runoff_inputs,
    routing_inputs     = routing_inputs,
    region             = 'West',
    product            = 'AORC',
    runoff_spinup_file = '/path/to/spinup/final_19801230_19801231.nc',
    slurm_cfg          = slurm_cfg,
    submit             = False,   # set True to submit
)

## 3. Forecast Run

In [ ]:
forecast_runoff_inputs = {
    **runoff_inputs,
    'init_file': '/path/to/initial_state.nc',     # runoff initial condition
}

forecast_routing_inputs = {
    **routing_inputs,
    'ini_flag': 1,
    'ini_file': '/path/to/snapshot_2019-12-31.nc',  # routing initial condition
}

setup_forecast(
    proj_root      = '/scratch/gpfs/user/MyForecast',
    start_date     = '2020-03-01',
    end_date       = '2020-03-15',
    runoff_inputs  = forecast_runoff_inputs,
    routing_inputs = forecast_routing_inputs,
    region         = 'West',
    product        = 'AORC',
    slurm_cfg      = slurm_cfg,
    submit         = False,
)

## 4. Spinup (repeat first year N times)

In [ ]:
setup_spinup(
    proj_root          = '/scratch/gpfs/user/MyProject',
    spinup_year        = 2002,
    n_cycles           = 5,
    runoff_inputs      = runoff_inputs,
    routing_inputs     = routing_inputs,
    region             = 'West',
    product            = 'AORC',
    runoff_spinup_file = '/path/to/spinup/final_19801230_19801231.nc',
    slurm_cfg          = slurm_cfg,
    submit             = False,
)